<a href="https://colab.research.google.com/github/dominiksakic/NETworkingMay/blob/main/27_machine_translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip
!unzip -q spa-eng.zip

--2025-05-31 00:20:48--  http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.69.207, 108.177.96.207, 108.177.119.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.69.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2638744 (2.5M) [application/zip]
Saving to: ‘spa-eng.zip’

spa-eng.zip         100%[===================>]   2.52M  3.78MB/s    in 0.7s    

2025-05-31 00:20:49 (3.78 MB/s) - ‘spa-eng.zip’ saved [2638744/2638744]



In [2]:
text_file = "spa-eng/spa.txt"
with open(text_file) as f:
  lines = f.read().split("\n")[:-1]
text_pairs = []

for line in lines:
  english, spanish = line.split("\t")
  spanish = "[start] " + spanish + " [end]"
  text_pairs.append((english, spanish))

In [3]:
import random
random.shuffle(text_pairs)
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples:num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples:]

In [4]:
# Vectorize the data
import tensorflow as tf
from tensorflow.keras import layers
import string
import re

# Handle special char not covered by strings.punctuation
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
  lowercase = tf.strings.lower(input_string)
  return tf.strings.regex_replace(
      lowercase, f"[{re.escape(strip_chars)}]", "")


vocab_size = 15000
sequence_length = 20

source_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
    )

target_vectorization = layers.TextVectorization(
  max_tokens=vocab_size,
  output_mode="int",
  # Off set by one by one step during training
  output_sequence_length=sequence_length + 1,
  standardize=custom_standardization,
)

train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
source_vectorization.adapt(train_english_texts)
target_vectorization.adapt(train_spanish_texts)

In [7]:
# Create dataset
batch_size = 64

def format_dataset(eng, spa):
  eng = source_vectorization(eng)
  spa = target_vectorization(spa)
  # Why do we keep the original spanish sentence?
  return({
      "english":eng,
      "spanish":spa[:,:-1] # remove last token, to keep same len
  }), spa[:, 1:] # one token ahead

def make_dataset(pairs):
  eng_texts, spa_texts = zip(*pairs)
  eng_texts = list(eng_texts)
  spa_texts = list(spa_texts)
  dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
  dataset = dataset.batch(batch_size)
  dataset = dataset.map(format_dataset, num_parallel_calls=4)
  return dataset.shuffle(2048).prefetch(16).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [10]:
for inputs, targets in train_ds.take(1):
  print(f"inputs['english'].shape: {inputs['english'].shape}")
  print(f"inputs['spanish'].shape: {inputs['spanish'].shape}")
  print(f"targets.shape: {targets.shape}")

inputs['english'].shape: (64, 20)
inputs['spanish'].shape: (64, 20)
targets.shape: (64, 20)


In [17]:
# Seq-to-Seq via RNNs


"""
Input into Encoder: "How is Yoda doing today"

Encoder RNN

Final output vector/final internal state  => Initial State of ...


Input into the Decoder : [start], Wie, geht, es, Yoda, heute

... Initial State -> Decoder RNN

Output: Wie, geht, es , Yoda, heute, [end]

"""

from tensorflow import keras
from tensorflow.keras import layers

embed_dim = 256
latent_dim = 1024

source = keras.Input(shape=(None,), dtype="int64", name="english")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)
encoded_source = layers.Bidirectional(
    layers.GRU(latent_dim), merge_mode="sum")(x)

past_target = keras.Input(shape=(None,), dtype="int64", name="spanish")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(past_target)
decoder_gru = layers.GRU(latent_dim, return_sequences=True)
x = decoder_gru(x, initial_state=encoded_source)
x = layers.Dropout(0.5)(x)
target_next_step = layers.Dense(vocab_size, activation="softmax")(x)
seq2seq_rnn = keras.Model([source, past_target], target_next_step)

In [18]:
seq2seq_rnn.compile(
  optimizer="rmsprop",
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"])

"""
How to evaluate the performance of a Seq-to-Seq model?
BLEU Score - a metric that evlautes  the entire generated sequences.
"""
seq2seq_rnn.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 224s 167ms/step - accuracy: 0.1527 - loss: 5.2632 - val_accuracy: 0.1563 - val_loss: 3.8669
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 257s 166ms/step - accuracy: 0.1609 - loss: 3.8693 - val_accuracy: 0.1890 - val_loss: 3.2437
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 261s 166ms/step - accuracy: 0.1862 - loss: 3.3176 - val_accuracy: 0.2092 - val_loss: 2.8640
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 215s 165ms/step - accuracy: 0.2033 - loss: 2.9496 - val_accuracy: 0.2232 - val_loss: 2.6184
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 215s 165ms/step - accuracy: 0.2170 - loss: 2.6626 - val_accuracy: 0.2338 - val_loss: 2.4416
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 215s 165ms/step - accuracy: 0.2286 - loss: 2.4301 - val_accuracy: 0.2411 - val_loss: 2.3127
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 221s 170ms/step - accuracy: 0.2386 - loss: 2.2396 - val_accuracy: 0.2474 - val_loss: 2.2174
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 215s 165ms/step - ac

In [19]:
# Lets play around with the model!
import numpy as np
spa_vocab = target_vectorization.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))
max_decoded_sentence_length = 20

def decode_sequence(input_sentence):
  tokenized_input_sentence = source_vectorization([input_sentence])
  decoded_sentence = "[start]"
  for i in range(max_decoded_sentence_length):
    tokenized_target_sentence = target_vectorization([decoded_sentence])
    next_token_predictions = seq2seq_rnn.predict(
        [tokenized_input_sentence,
         tokenized_target_sentence])
    # Sample strategy
    sampled_token_index = np.argmax(next_token_predictions[0, i, :])
    # Convert int to Text
    sampled_token = spa_index_lookup[sampled_token_index]
    decoded_sentence += " " + sampled_token
    if sampled_token == "[end]":
      break

  return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(20):
  input_sentence = random.choice(test_eng_texts)
  print("-")
  print(input_sentence)
  print(decode_sequence(input_sentence))

-
The only flavor ice cream that Tom eats is vanilla.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
[start] el único que se [UNK] está en el que tom está [UNK] [end]
-
She did the work carefully.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
[start] ella hizo el trabajo con el trabajo [end]
-